In [1]:

!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd
import plotly.express as px

# Load real credit-risk data
dataset = fetch_ucirepo(id=144)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

print("Number of records:", len(X))
print("Number of features:", X.shape[1])
print("\nFirst five records:")
display(X.head())

# Interactive visualization
fig = px.histogram(
    X,
    x="Attribute13",
    nbins=25,
    title="Age Distribution of Credit Applicants",
    labels={
        "Attribute13": "Age (years)",
        "count": "Number of applicants"
    },
    color_discrete_sequence=["#5B8DEF"]
)

fig.update_layout(
    template="plotly_dark",
    bargap=0.08
)

fig.show()


Number of records: 1000
Number of features: 20

First five records:


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import plotly.express as px

# Target: 1 = good credit, 2 = bad credit
target = y.iloc[:, 0]
target = (target == 2).astype(int)

# Separate numerical and categorical features
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2,
    random_state=42, stratify=target
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
display(results_df)

# Interactive comparison chart
chart_data = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "ROC AUC"],
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    chart_data,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Credit Risk Model Performance",
    text_auto=".3f",
    template="plotly_dark"
)

fig.update_layout(yaxis_range=[0, 1])
fig.show()


,Model,Accuracy,ROC AUC
0,Logistic Regression,0.78,0.804048
1,Random Forest,0.76,0.783750


In [3]:

import numpy as np
import pandas as pd
import plotly.express as px

# Use the existing test set and trained models
fairness_results = []

# Exploratory age groups: under 35 versus 35+
age_groups = np.where(
    X_test["Attribute13"].to_numpy() < 35,
    "Under 35",
    "35 and older"
)

for model_name, model in trained_models.items():
    predictions = model.predict(X_test)

    for group in ["Under 35", "35 and older"]:
        mask = age_groups == group

        actual = y_test.to_numpy()[mask]
        predicted = predictions[mask]

        negative_mask = actual == 0

        false_positive_rate = (
            np.mean(predicted[negative_mask] == 1)
            if negative_mask.sum() > 0
            else np.nan
        )

        fairness_results.append({
            "Model": model_name,
            "Age Group": group,
            "Sample Size": len(actual),
            "Actual High-Risk Rate": actual.mean(),
            "Predicted High-Risk Rate": predicted.mean(),
            "False Positive Rate": false_positive_rate
        })

fairness_df = pd.DataFrame(fairness_results)

display(fairness_df.round(3))

# Interactive fairness comparison
fig = px.bar(
    fairness_df,
    x="Age Group",
    y="Predicted High-Risk Rate",
    color="Model",
    barmode="group",
    text_auto=".1%",
    title="Predicted Credit Risk by Age Group",
    template="plotly_dark",
    hover_data=["Sample Size", "Actual High-Risk Rate",
                "False Positive Rate"]
)

fig.update_layout(
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1]
)

fig.show()


,Model,Age Group,Sample Size,Actual High-Risk Rate,Predicted High-Risk Rate,False Positive Rate
0,Logistic Regression,Under 35,105,0.39,0.314,0.141
1,Logistic Regression,35 and older,95,0.20,0.158,0.092
2,Random Forest,Under 35,105,0.39,0.267,0.141
3,Random Forest,35 and older,95,0.20,0.105,0.053


In [4]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

# Final research dashboard
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Model Accuracy",
        "Predicted High-Risk Rate",
        "False Positive Rate"
    )
)

# Chart 1: Model accuracy
for _, row in results_df.iterrows():
    fig.add_trace(
        go.Bar(
            x=[row["Model"]],
            y=[row["Accuracy"]],
            name=row["Model"],
            showlegend=False
        ),
        row=1, col=1
    )

# Chart 2: Predicted risk by age group
for model_name in fairness_df["Model"].unique():
    data = fairness_df[fairness_df["Model"] == model_name]

    fig.add_trace(
        go.Bar(
            x=data["Age Group"],
            y=data["Predicted High-Risk Rate"],
            name=model_name
        ),
        row=1, col=2
    )

# Chart 3: False positive rates
for model_name in fairness_df["Model"].unique():
    data = fairness_df[fairness_df["Model"] == model_name]

    fig.add_trace(
        go.Bar(
            x=data["Age Group"],
            y=data["False Positive Rate"],
            name=model_name,
            showlegend=False
        ),
        row=1, col=3
    )

fig.update_layout(
    title="Responsible AI: Credit Risk & Fairness Dashboard",
    template="plotly_dark",
    height=550,
    width=1400,
    barmode="group"
)

fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

# Save dashboard as an interactive HTML file
fig.write_html(
    "responsible_ai_dashboard.html",
    include_plotlyjs=True,
    full_html=True
)

print("Interactive dashboard saved successfully!")


Interactive dashboard saved successfully!
